In [2]:
import pyspark
from pyspark.sql import SparkSession

In [5]:
spark = SparkSession.builder \
        .master("local[*]") \
        .appName("Homework6") \
        .getOrCreate()

In [6]:
# Question 1
spark.version

'4.1.1'

In [11]:
df = spark.read \
     .parquet("yellow_tripdata_2025-11.parquet")

In [9]:
df.repartition(4).write.parquet("output/yellow_repartitioned")

In [ ]:
# Question 2
import os

folder = "output/yellow_repartitioned"
files = [f for f in os.listdir(folder) if f.endswith(".parquet")]

for f in files:
    size = os.path.getsize(os.path.join(folder, f))
    print(f"{f}: {size / (1024**2):.2f} MB")

# Average size
sizes = [os.path.getsize(os.path.join(folder, f)) for f in files]
avg_mb = (sum(sizes) / len(sizes)) / (1024**2)
print(f"\nAverage size: {avg_mb:.2f} MB")

In [22]:
# Question 3
from pyspark.sql.functions import col, day, month

num_trips = df \
    .filter((day(col("tpep_pickup_datetime")) == 15) & (month(col("tpep_pickup_datetime")) == 11)) \
    .count()

print(num_trips)

162604


In [31]:
# Question 4
from pyspark.sql.functions import unix_timestamp

df.withColumn("duration_hours", 
    (unix_timestamp("tpep_dropoff_datetime") - unix_timestamp("tpep_pickup_datetime")) / 3600
).orderBy("duration_hours", ascending=False) \
 .select("tpep_pickup_datetime", "tpep_dropoff_datetime", "duration_hours") \
 .limit(1).show()

+--------------------+---------------------+-----------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|   duration_hours|
+--------------------+---------------------+-----------------+
| 2025-11-26 20:22:12|  2025-11-30 15:01:00|90.64666666666666|
+--------------------+---------------------+-----------------+



In [33]:
df_zones = spark.read \
           .option("header", True) \
           .csv("taxi_zone_lookup.csv")

In [44]:
# Question 6
df.registerTempTable("taxis")
df_zones.registerTempTable("zones")
result = spark.sql(
    """
    SELECT t.PULocationID, z.Zone, COUNT(*)
    FROM taxis t
    JOIN zones z
    ON t.PULocationID = z.LocationID
    GROUP BY t.PULocationID, z.Zone
    ORDER BY COUNT(*)
    """
).show()

+------------+--------------------+--------+
|PULocationID|                Zone|count(1)|
+------------+--------------------+--------+
|         105|Governor's Island...|       1|
|          84|Eltingville/Annad...|       1|
|           5|       Arden Heights|       1|
|         187|       Port Richmond|       3|
|         204|   Rossville/Woodrow|       4|
|         199|       Rikers Island|       4|
|         111| Green-Wood Cemetery|       4|
|         109|         Great Kills|       4|
|           2|         Jamaica Bay|       5|
|         251|         Westerleigh|      12|
|         176|             Oakwood|      14|
|         172|New Dorp/Midland ...|      14|
|          59|        Crotona Park|      14|
|         245|       West Brighton|      14|
|         253|       Willets Point|      15|
|          27|Breezy Point/Fort...|      16|
|         206|Saint George/New ...|      17|
|          30|       Broad Channel|      18|
|         156|     Mariners Harbor|      21|
|         